In [1]:
%pip install gliner bert_score sentence_transformers osmnx


[notice] A new release of pip is available: 25.0.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [ ]:
import json
import pickle
import numpy as np
import osmnx as ox
import networkx as nx
import spacy
import torch
import re
from tqdm import tqdm
from gliner import GLiNER
from sentence_transformers import SentenceTransformer
from eval.evaluation import prompt_based_route_evaluator
from routing.synset import OSMSemanticBridge
from scipy.special import softmax

# --- 1. GLiNER MANAGER (Judge & Extractor) ---

class GLiNERManager:
    def __init__(self, model_id="urchade/gliner_medium-v2.1"):
        print(f"\nInitializing GLiNER-2 Judge: {model_id}...")
        # GLiNER fits easily in memory and runs fast on CPU
        self.model = GLiNER.from_pretrained(model_id)
        # Define the labels for intent extraction
        self.labels = ["avoid_feature", "preferred_feature"]

    def extract_intent(self, user_prompt, tag_schema):
        """
        Uses GLiNER to identify preferred and avoided features, 
        then maps them to the OSM tag schema.
        """
        entities = self.model.predict_entities(user_prompt, self.labels, threshold=0.55)
        
        intent = {"avoid_tags": {}, "prefer_tags": {}}
        
        # Mapping extracted entities to schema categories via simple string matching
        # Note: In a production setting, you'd use the bridge/synset logic here.
        for ent in entities:
            text = ent["text"].lower()
            label = "avoid_tags" if ent["label"] == "avoid_feature" else "prefer_tags"
            
            # Simple heuristic mapping for the experiment
            for cat, values in tag_schema["discrete"].items():
                for val in values:
                    if val in text or text in val:
                        if cat not in intent[label]:
                            intent[label][cat] = set()
                        intent[label][cat].add(val)
        
        # Convert sets to lists for JSON compatibility
        return {k: {cat: list(vals) for cat, vals in v.items()} for k, v in intent.items()}

    def judge_candidates(self, user_prompt, candidates, G, baseline_dist):
        """
        Scores routes by detecting how many desired features are present 
        along the path edges vs avoided features.
        """
        if not candidates: return None
        if len(candidates) == 1: return candidates[0]

        # Get what the user specifically asked for
        entities = self.model.predict_entities(user_prompt, self.labels, threshold=0.55)
        targets = [e["text"].lower() for e in entities if e["label"] == "preferred_feature"]
        avoids = [e["text"].lower() for e in entities if e["label"] == "avoid_feature"]

        scores = []
        for path in candidates:
            # Aggregate all unique highway/landuse tags in the candidate path
            path_tags = []
            for u, v in zip(path[:-1], path[1:]):
                data = G.get_edge_data(u, v)[0]
                hw = data.get('highway', 'unknown')
                path_tags.append(hw[0] if isinstance(hw, list) else hw)
            
            path_description = " ".join(set(path_tags)).lower()
            
            # Score based on keyword matches in the route's topological features
            current_score = 0
            for t in targets:
                if t in path_description: current_score += 10
            for a in avoids:
                if a in path_description: current_score -= 20
            
            # Efficiency Penalty: Slightly favor shorter paths if scores are tied
            dist = sum(G.get_edge_data(u, v)[0].get('length', 0) for u, v in zip(path[:-1], path[1:]))
            dist_ratio = dist / baseline_dist if baseline_dist > 0 else 1.0
            current_score -= dist_ratio
            
            scores.append(current_score)

        return candidates[np.argmax(scores)]

# --- 2. SMC GENERATOR (Remains focused on particle filtering) ---

class SMCRouteGenerator:
    def __init__(self, G, bridge, st_model, n_particles=100):
        self.G = G
        self.bridge = bridge
        self.st_model = st_model
        self.n_particles = n_particles
        self.intent_cache = {}
            
    def _get_distance(self, node, target):
        n1 = self.G.nodes[node]
        n2 = self.G.nodes[target]
        return ox.distance.great_circle(n1['y'], n1['x'], n2['y'], n2['x'])

    def _calculate_edge_weight(self, edge_data, intent):
        multiplier = 1.0
        for cat in self.bridge.categories:
            val = edge_data.get(cat)
            if not val: continue
            val = val[0] if isinstance(val, list) else val
            
            # Accessing list from JSON-ified intent
            if val in intent["avoid_tags"].get(cat, []):
                multiplier *= 0.1 
            if val in intent["prefer_tags"].get(cat, []):
                multiplier *= 10.0
        return multiplier

    def find_routes(self, start_node, end_node, prompt, intent, n_candidates=3, max_steps=500):
        try:
            anchor_path = nx.astar_path(self.G, start_node, end_node, weight='length')
            anchor_nodes = set(anchor_path)
        except nx.NetworkXNoPath:
            return []

        particles = [[start_node, [start_node], 1.0, False] for _ in range(self.n_particles)]
        unique_completed_paths = {}

        for step in range(max_steps):
            active = [i for i, p in enumerate(particles) if not p[3]]
            if not active or len(unique_completed_paths) >= n_candidates:
                break

            for i in active:
                curr_node = particles[i][0]
                neighbors = list(self.G.neighbors(curr_node))
                if not neighbors:
                    particles[i][2] = 0
                    continue
                
                dists = np.array([self._get_distance(n, end_node) for n in neighbors])
                on_anchor = np.array([50.0 if n in anchor_nodes else 0.0 for n in neighbors])
                
                scores = -(dists / 100.0) + on_anchor 
                probs = softmax(scores) 
                
                next_node = np.random.choice(neighbors, p=probs)
                edge_data = self.G.get_edge_data(curr_node, next_node)
                data = list(edge_data.values())[0] if isinstance(edge_data, dict) else edge_data
                
                particles[i][2] *= self._calculate_edge_weight(data, intent)
                particles[i][0] = next_node
                particles[i][1].append(next_node)
                
                if next_node == end_node:
                    particles[i][3] = True
                    path_tuple = tuple(particles[i][1])
                    unique_completed_paths[path_tuple] = particles[i][2]

            # Resampling
            if step % 20 == 0:
                weights = np.array([p[2] for p in particles])
                if weights.sum() > 0:
                    weights /= weights.sum()
                    idx = np.random.choice(range(self.n_particles), size=self.n_particles, p=weights)
                    particles = [[particles[j][0], list(particles[j][1]), 1.0, particles[j][3]] for j in idx]

        sorted_paths = sorted(unique_completed_paths.items(), key=lambda x: x[1], reverse=True)
        return [list(path) for path, weight in sorted_paths[:n_candidates]]

# --- 3. MAIN EXPERIMENT SCRIPT ---

if __name__ == "__main__":
    # Load Data (Assuming paths are correct for your environment)
    with open("research/pioneer_valley_v2.pkl", "rb") as f:
        G = pickle.load(f)
    with open("research/synthetic_dataset.jsonl", "r") as f:
        data = [json.loads(line) for line in f]

    tag_schema = {
        "continuous": {
            "maxspeed_imputed": {"min": 10, "max": 65, "unit": "mph"},
            "lanes": {"min": 1, "max": 6},
            "length": {"min": 2, "max": 6845}
        },
        "discrete": {
            "highway": ['residential', 'trunk', 'secondary', 'tertiary', 'primary', 'motorway_link', 
                        'unclassified', 'secondary_link', 'motorway', 'tertiary_link', 'primary_link', 'trunk_link'],
            "access": ["yes", "no"],
            "bridge": ["yes", "viaduct"],
            "junction": ["roundabout", "jughandle"],
            "ref": ['US 20', 'US 5', 'MA 9', 'MA 10', 'MA 66', 'MA 187', 'US 202', 'I 91', 'I 90', 'MA 116']
        }
    }

    st_model = SentenceTransformer('all-MiniLM-L6-v2')
    bridge = OSMSemanticBridge(tag_schema, st_model, threshold=0.80)
    
    # Initialize GLiNER instead of LLM
    gliner_manager = GLiNERManager()
    smc_generator = SMCRouteGenerator(G, bridge, st_model, n_particles=100)
    
    gen_routes, gen_prompts = [], []

    print("\n--- Running Neurosymbolic Experiment (Mode: SMC + GLiNER Judge) ---")
    
    for idx, item in enumerate(tqdm(data[100:200])):
        try:
            s_pt = ox.geocoder.geocode(f"{item['start']}, MA, USA")
            e_pt = ox.geocoder.geocode(f"{item['end']}, MA, USA")
            s_node = ox.distance.nearest_nodes(G, X=s_pt[1], Y=s_pt[0])
            e_node = ox.distance.nearest_nodes(G, X=e_pt[1], Y=e_pt[0])
            
            if not nx.has_path(G, s_node, e_node): continue

            # 1. GLiNER Intent Extraction
            intent = gliner_manager.extract_intent(item['prompt'], tag_schema)
            
            # 2. SMC Candidate Generation
            candidates = smc_generator.find_routes(s_node, e_node, item['prompt'], intent)
            
            if candidates:
                anchor_path = nx.astar_path(G, s_node, e_node, weight='length')
                base_dist = sum(G.get_edge_data(u,v)[0].get('length',0) for u,v in zip(anchor_path[:-1], anchor_path[1:]))

                # 3. GLiNER Selection
                best_route = gliner_manager.judge_candidates(item['prompt'], candidates, G, base_dist)
                
                gen_routes.append(best_route) 
                gen_prompts.append(item['prompt'])
                
        except Exception as e:
            continue

    if gen_routes:
        evaluator = prompt_based_route_evaluator(G, gen_prompts, gen_routes, bridge)
        results = evaluator.evaluate_method()
        print("\n--- Final Consolidated Metrics ---")
        for metric, value in results.items():
            print(f"{metric:30}: {value:.4f}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Bridge: Building Schema-Grounded Index for categories: ['highway', 'access', 'bridge', 'junction', 'ref']

Initializing GLiNER-2 Judge: urchade/gliner_medium-v2.1...


/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/huggingface_hub/utils/_validators.py:189: UserWarning: The `resume_download` argument is deprecated and ignored in `snapshot_download`. Downloads always resume whenever possible.
  warnings.warn(


Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

...

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MPNetModel LOAD REPORT from: sentence-transformers/multi-qa-mpnet-base-dot-v1
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



--- Final Consolidated Metrics ---
avg_path_validity             : 1.0000
avg_deviation_penalty         : 1.4678
avg_constraint_satisfaction   : 0.4420
avg_semantic_alignment_f1     : 0.7917
